# 🔥 Project IRIS: Heretic 7B Model Decensoring Pipeline

This Google Colab notebook runs [**Heretic**](https://github.com/p-e-w/heretic) to automatically abliterate refusal / censorship directions from **`Qwen/Qwen2.5-Coder-7B-Instruct`** and exports the resulting uncensored weights into **GGUF format (`Q4_K_M`)** for deployment in **Ollama** and **Project IRIS**.

### ⚠️ Important: Hardware Acceleration Setup
Before running, ensure hardware acceleration is enabled:
1. Go to **Runtime → Change runtime type**.
2. Select **T4 GPU** (free tier) or **A100 / L4** (Colab Pro).
3. Click **Save**.

### 1. Verify GPU Availability
Check that a CUDA-compatible GPU is attached.

In [1]:
!nvidia-smi

Mon Oct 12 16:47:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 2. Install Heretic & Acceleration Dependencies
Install `heretic-llm`, PyTorch, bitsandbytes (for 4-bit quantization during optimization), and Optuna.

In [2]:
# Install heretic and core libraries
!pip install -U heretic-llm bitsandbytes accelerate optuna transformers huggingface_hub peft

# Ensure llama.cpp requirements for GGUF conversion are ready
!pip install gguf sentencepiece

  Using cached optuna-5.0.0-py3-none-any.whl.metadata (14 kB)
  Using cached huggingface_hub-2.2.0-py3-none-any.whl.metadata (16 kB)


### 3. Configure Model (7B) & Optimization Parameters
Using **Qwen/Qwen2.5-Coder-7B-Instruct** for superior reasoning, function calling, and zero refusals.

In [3]:
import os

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"   # 7B Target HF model
OUTPUT_DIR = "/content/heretic_7b_output"     # Destination for abliterated weights
N_TRIALS = 35                                 # 35 trials is fast and effective for 7B on T4
QUANTIZATION = "bnb_4bit"                     # 4-bit takes ~5.5GB VRAM (fits easily in 16GB T4)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Target Model: {MODEL_ID}")
print(f"Optimization Trials: {N_TRIALS}")
print(f"Quantization: {QUANTIZATION}")

Target Model: Qwen/Qwen2.5-Coder-7B-Instruct
Optimization Trials: 35
Quantization: bnb_4bit


### 4. Run Heretic Abliteration & Optimization
Heretic will:
1. Benchmark the GPU for optimal batch size.
2. Extract harmless vs harmful activation residuals.
3. Run Optuna TPE to find the best abliteration kernel and LoRA weights.
4. Merge and export the decensored 7B model weights into `OUTPUT_DIR`.

In [4]:
!heretic \
    --model "{MODEL_ID}" \
    --quantization "{QUANTIZATION}" \
    --n-trials {N_TRIALS} \
    --export-strategy merge \
    --export-path "{OUTPUT_DIR}"

█░█░█▀▀░█▀▄░█▀▀░▀█▀░█░█▀▀  v1.4.0
█▀█░█▀▀░█▀▄░█▀▀░░█░░█░█░░  https://heretic-project.org
▀░▀░▀▀▀░▀░▀░▀▀▀░░▀░░▀░▀▀▀  https://github.com/p-e-w/heretic

usage: heretic [-h] [--model str] [--model-commit {str,null}]
               [--evaluate-model {str,null}]
               [--collect-reproducibles {str,null}] [--reproduce {str,null}]
               [--dtypes list[str]] [--quantization {NONE,BNB_4BIT}]
               [--device-map {str,Dict[str,{int,str}]}]
               [--max-memory {Dict[str,str],null}]
               [--offload-outputs-to-cpu | --no-offload-outputs-to-cpu]
               [--batch-size int] [--max-batch-size int]
               [--max-response-length int] [--response-prefix {str,null}]
               [--chain-of-thought-skips list[tuple[str,str]]]
               [--print-responses | --no-print-responses]
               [--print-residual-geometry | --no-print-residual-geometry]
               [--plot-residuals | --no-plot-residuals]
               [--residual-plot-path

### 5. Convert Abliterated 7B Weights to GGUF Format (`Q4_K_M`)
We clone `llama.cpp` and quantize the 7B model to `Q4_K_M` (~4.4 GB), allowing it to fit into RAM.

In [5]:
%cd /content
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
%cd /content/llama.cpp
!make -j llama-quantize || cmake -B build && cmake --build build --config Release -t llama-quantize
!pip install -r requirements.txt

# Step A: Convert to f16 intermediate GGUF
GGUF_F16 = "/content/iris-qwen2.5-coder-7b-heretic.f16.gguf"
!python3 convert_hf_to_gguf.py "{OUTPUT_DIR}" --outfile "{GGUF_F16}" --outtype f16

# Step B: Quantize to Q4_K_M for optimal 8GB RAM local performance (~4.4 GB)
GGUF_FINAL = "/content/iris-qwen2.5-coder-7b-heretic.Q4_K_M.gguf"
if os.path.exists("./build/bin/llama-quantize"):
    !./build/bin/llama-quantize "{GGUF_F16}" "{GGUF_FINAL}" Q4_K_M
elif os.path.exists("./llama-quantize"):
    !./llama-quantize "{GGUF_F16}" "{GGUF_FINAL}" Q4_K_M
else:
    # Fallback to direct q8_0 conversion if quantize binary build is skipped
    !python3 convert_hf_to_gguf.py "{OUTPUT_DIR}" --outfile "{GGUF_FINAL}" --outtype q8_0

print(f"Conversion complete! Final 7B GGUF created at: {GGUF_FINAL}")

/content
Cloning into 'llama.cpp'...
remote: Enumerating objects: 4132, done.
remote: Counting objects: 100% (4132/4132), done.
remote: Compressing objects: 100% (3426/3426), done.
remote: Total 4132 (delta 727), reused 2871 (delta 624), pack-reused 0 (from 0)
Receiving objects: 100% (4132/4132), 36.65 MiB | 11.41 MiB/s, done.
Resolving deltas: 100% (727/727), done.
Updating files: 100% (3774/3774), done.
/content/llama.cpp
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check 

INFO:hf-to-gguf:Loading model: heretic_7b_output
Traceback (most recent call last):
  File "/content/llama.cpp/conversion/base.py", line 1452, in load_hparams
    config = AutoConfig.from_pretrained(dir_model, trust_remote_code=False).to_dict()
             ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/transformers/models/auto/configuration_auto.py", line 1380, in from_pretrained
    raise ValueError(
    ...<3 lines>...
    )
ValueError: Unrecognized model in /content/heretic_7b_output. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: aimv2, aimv2_vision_model, albert, align, altclip, apertus, arcee, aria, aria_text, audio-spectrogram-transformer, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, bitnet, blenderbot, blenderbot-small, blip, blip-2, blip_2_qformer, bloom, blt, bridgetower, bros, camembert, c

### 6. Download the 7B Model File
You can save the model directly to your Google Drive or download it to your local machine.

In [1]:
from google.colab import drive
import shutil

try:
    drive.mount('/content/drive')
    shutil.copy(GGUF_FINAL, '/content/drive/MyDrive/iris-qwen2.5-coder-7b-heretic.Q4_K_M.gguf')
    print("Successfully saved 7B GGUF to your Google Drive under MyDrive!")
except Exception as e:
    print("Google Drive mount skipped. You can download via the file explorer on the left or colab files download:")
    from google.colab import files
    files.download(GGUF_FINAL)

Mounted at /content/drive
Google Drive mount skipped. You can download via the file explorer on the left or colab files download:


NameError: name 'GGUF_FINAL' is not defined